# Design Decisions: Loading spectra

This notebook illustrates some of the design decision for the wrap_xspec.SpectrumLoader. 

In [1]:
import xspec
import wrap_xspec as wx

from IPython.core.interactiveshell import InteractiveShell

Below I define some filenames [Note that I will change the naming convension to match that of the xspec.Spectrum class init function]. 

The structure of the files is as such:
````
- Tutorials
    |- This_notebook.ipynb
    |- InputData
        |- Dash2_acisf14571_grpSNR3.pi
        |- Dash2_acisf14571.rmf
        ...
````

As you can see, the files are not in the current working directories. 

However, the `Dash2_acisf14571_grpSNR3.pi` files have header keywords that lists the name of the associated files (rmf, arf, back). 

In [2]:
filenames={
    'pi'  : 'Dash2_acisf14571_grpSNR3.pi',
    'back': None,  ##'Dash2_acisf14571_bkg.pi',
    'rmf' : 'Dash2_acisf14571.rmf',
    'arf' : 'Dash2_acisf14571.corr.arf',
    'path': 'InputData'
}

So we can use the paths of each file in the call to the native xspec.Spectrum call, as I show below. 

But you will notice all of the warnings and error messages that pops out of `xspec`. 

In [3]:
xspec.AllData.clear()

s = xspec.Spectrum('InputData/'+filenames['pi'],
               backFile=None,
               arfFile='InputData/'+filenames['arf'],
               respFile='InputData/'+filenames['rmf']
               )



 response file  read skipped

 background file  read skipped

1 spectrum  in use
 
Spectral Data File: InputData/Dash2_acisf14571_grpSNR3.pi  Spectrum 1
Net count rate (cts/s) for Spectrum:1  8.855e-03 +/- 4.231e-04
 Assigned to Data Group 1 and Plot Group 1
  Noticed Channels:  1-399
  Telescope: CHANDRA Instrument: ACIS  Channel Type: PI
  Exposure Time: 4.946e+04 sec
 Using fit statistic: chi
 No response loaded.

***Warning!  One or more spectra are missing responses,
               and are not suitable for fit.
Response successfully loaded.
Arf successfully loaded.
Spectrum 1  Spectral Data File: InputData/Dash2_acisf14571_grpSNR3.pi
Net count rate (cts/s) for Spectrum:1  8.855e-03 +/- 4.231e-04
 Assigned to Data Group 1 and Plot Group 1
  Noticed Channels:  1-399
  Telescope: CHANDRA Instrument: ACIS  Channel Type: PI
  Exposure Time: 4.946e+04 sec
 Using fit statistic: chi
 Using Response (RMF) File            InputData/Dash2_acisf14571.rmf for Source 1
 Using Auxiliary Respons

Error: cannot read response file Dash2_acisf14571.rmf

***XSPEC Error:  cannot open file named: Dash2_acisf14571_bkg.pi
Error: cannot read background file 


This is because `xspec.Spectrum` will **always** try to load the files from the header, even if the user included the path of the desired file (or None) in the Spectrum call. 

In this case, the file listed in the headers do not exist in the current directory (because they are in `InputData/`). 

Therefore, the error messages are misleading to a novice xspec user, even if everything is loaded properly, as can be shown from the call to `AllData.show()` below, that shows the current status of xspec

In [4]:
xspec.AllData.show()


1 file 1 spectrum 
Spectrum 1  Spectral Data File: InputData/Dash2_acisf14571_grpSNR3.pi
Net count rate (cts/s) for Spectrum:1  8.855e-03 +/- 4.231e-04
 Assigned to Data Group 1 and Plot Group 1
  Noticed Channels:  1-399
  Telescope: CHANDRA Instrument: ACIS  Channel Type: PI
  Exposure Time: 4.946e+04 sec
 Using fit statistic: chi
 Using Response (RMF) File            InputData/Dash2_acisf14571.rmf for Source 1
 Using Auxiliary Response (ARF) File  InputData/Dash2_acisf14571.corr.arf



Here's another example, where I input the path of an inexisting file to the background, and the wrong type of file to the arf. 

As you can see, xspec gives warning and errors messages (like above!) that this time are critical. But a novice user might not yet be able to make the difference. 

As you notice, the python code **does not crash**. 


In [5]:
xspec.AllData.clear()

s = xspec.Spectrum('InputData/'+filenames['pi'],
               backFile='FileThatDoesNotExist', ## <- inexisting file
               arfFile='InputData/'+filenames['rmf'], ## <- loading an existing file that is NOT an arf
               respFile='InputData38383/'+filenames['rmf']
               )


 response file  read skipped
Error: No response is assigned to source 1 for spectrum 1
Error: Requested arf file not loaded

 background file  read skipped

1 spectrum  in use
 
Spectral Data File: InputData/Dash2_acisf14571_grpSNR3.pi  Spectrum 1
Net count rate (cts/s) for Spectrum:1  8.855e-03 +/- 4.231e-04
 Assigned to Data Group 1 and Plot Group 1
  Noticed Channels:  1-399
  Telescope: CHANDRA Instrument: ACIS  Channel Type: PI
  Exposure Time: 4.946e+04 sec
 Using fit statistic: chi
 No response loaded.

***Warning!  One or more spectra are missing responses,
               and are not suitable for fit.

 background file  read skipped

 response file  read skipped
Spectrum 1 has no response for source 1
Spectrum 1  Spectral Data File: InputData/Dash2_acisf14571_grpSNR3.pi
Net count rate (cts/s) for Spectrum:1  8.855e-03 +/- 4.231e-04
 Assigned to Data Group 1 and Plot Group 1
  Noticed Channels:  1-399
  Telescope: CHANDRA Instrument: ACIS  Channel Type: PI
  Exposure Time: 4.94

Error: cannot read response file Dash2_acisf14571.rmf

***XSPEC Error:  cannot open file named: Dash2_acisf14571_bkg.pi
Error: cannot read background file 

***XSPEC Error:  cannot open file named: FileThatDoesNotExist.pha
Error: cannot read background file 
Error: cannot read response file InputData38383/Dash2_acisf14571.rmf


## Our solution

Our `wrap_xspec.SpectrumLoader` provides a stricter way to handle loading spectra, that novice user (and faculty who are interested in improving the clarity and reproducabiity of their students' work) will find useful. 

First, the function captures and hides the intermediate xspec messaging (it is available by using the verbose=True in the SpectrumLoader.xspec_load() call). 

It displays only the final status of the xspec.AllData. Here's an example below:

In [6]:
xspec.AllData.clear()

filenames={
    'pi'  : 'Dash2_acisf14571_grpSNR3.pi',
    'back': 'Dash2_acisf14571_bkg.pi',
    'rmf' : 'Dash2_acisf14571.rmf',
    'arf' : 'Dash2_acisf14571.corr.arf',
    'path': 'InputData'
}

my_spec1 = wx.SpectrumLoader(**filenames)

spec1 = my_spec1.xspec_load()

print(spec1.background)

[Wrap_xspec]: Current Status of xspec.AllData:


1 file 1 spectrum 
Spectrum 1  Spectral Data File: InputData/Dash2_acisf14571_grpSNR3.pi
Net count rate (cts/s) for Spectrum:1  8.410e-03 +/- 4.238e-04 (95.0 % total)
 Assigned to Data Group 1 and Plot Group 1
  Noticed Channels:  1-399
  Telescope: CHANDRA Instrument: ACIS  Channel Type: PI
  Exposure Time: 4.946e+04 sec
 Using fit statistic: chi
 Using Background File                InputData/Dash2_acisf14571_bkg.pi
  Background Exposure Time: 4.946e+04 sec
 Using Response (RMF) File            InputData/Dash2_acisf14571.rmf for Source 1
 Using Auxiliary Response (ARF) File  InputData/Dash2_acisf14571.corr.arf



Furthermore, `wrap_xspec.SpectrumLoader` is designed **to make python crash explitely** when something goes wrong for real. 


### Files that do not exist: 

It validates the input to the function, verify that the files listed actually exists even before touching xspec, and it validates the final status of xspec.AllData against the user inputs. 

It outputs useful python error messages. 

In [7]:
xspec.AllData.clear()

filenames={
    'pi'  : 'Dash2_acisf14571_grpSNR3.pi',
    'back': 'AFileThatDoesNotExist', # <- inexisting file
    'rmf' : 'Dash2_acisf14571.rmf',
    'arf' : 'Dash2_acisf14571.corr.arf',
    'path': 'InputData'
}

try:
    my_spec1 = wx.SpectrumLoader(**filenames)
except Exception:
    print('=========================================')
    print('This is the error message you will see: ')
    print('=========================================')
    print()
    InteractiveShell.instance().showtraceback()
    print()
    print('=========================================')


This is the error message you will see: 



FileNotFoundError: Background file not found at expected location: 'InputData/AFileThatDoesNotExist'

### Loading a file that is not a valid spectrum file

In the cell below, we load two pi files (without responses/arf/background). 

The second file is not a valid spectrum file (it is an arf). 

In [8]:
xspec.AllData.clear()

pi = ['InputData/'+filenames['pi'], 
      'InputData/'+filenames['arf']]
my_spec1 = wx.SpectrumLoader(pi)

try:
    spec1 = my_spec1.xspec_load(verbose=False)
except Exception:
    print('=========================================')
    print('This is the error message you will see: ')
    print('=========================================')
    print()
    InteractiveShell.instance().showtraceback()
    print()
    print('=========================================')

xspec.AllData.show()

Error: No response is assigned to source 1 for spectrum 1
Error: Requested arf file not loaded
This is the error message you will see: 



RuntimeError: [wrap_xspec]: PyXspec failed to load spectrum at index 1 ('InputData/Dash2_acisf14571.corr.arf'). Please make sure that InputData/Dash2_acisf14571.corr.arf is a valid spectrum file. Underlying error: PyCapsule_New called with null pointer



1 file 1 spectrum 
Spectrum 1  Spectral Data File: InputData/Dash2_acisf14571_grpSNR3.pi
Net count rate (cts/s) for Spectrum:1  8.855e-03 +/- 4.231e-04
 Assigned to Data Group 1 and Plot Group 1
  Noticed Channels:  1-399
  Telescope: CHANDRA Instrument: ACIS  Channel Type: PI
  Exposure Time: 4.946e+04 sec
 Using fit statistic: chi
 No response loaded.



### One of the other files (rmf/arf/back) is not a valid file

In [9]:
xspec.AllData.clear()

filenames={
    'pi'  : 'InputData/Dash2_acisf14571_grpSNR3.pi',
    'back': 'InputData/Dash2_acisf14571_bkg.pi',
    'rmf' : 'InputData/Dash2_acisf14571.rmf',
    'arf' : 'InputData/Dash2_acisf14571.corr.arf',
}

my_spec1 = wx.SpectrumLoader(filenames['pi'], 
                             rmf=filenames['rmf'], 
                             arf=filenames['arf'], 
                             back=filenames['arf'] # <- Loading arf instead of a background file
                             )

try:
    spec1 = my_spec1.xspec_load(verbose=False)
except Exception:
    print('=========================================')
    print('This is the error message you will see: ')
    print('=========================================')
    print()
    InteractiveShell.instance().showtraceback()
    print()
    print('=========================================')


This is the error message you will see: 



ValueError: [wrap_xspec] Error at index 0: Loader expected background 'InputData/Dash2_acisf14571.corr.arf', but XSpec reports no background is loaded. Check that 'InputData/Dash2_acisf14571.corr.arf' is really a background file. 